# Auth Log Anomaly Detection

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

log_data = """
May 15 12:01:01 server sshd[2222]: Failed password for invalid user admin from 192.168.0.10 port 22 ssh2
May 15 12:01:02 server sshd[2223]: Failed password for root from 192.168.0.10 port 22 ssh2
May 15 03:41:45 server sshd[2230]: Accepted password for user1 from 203.0.113.42 port 22 ssh2
May 15 03:45:01 server sshd[2231]: Failed password for user1 from 203.0.113.42 port 22 ssh2
May 15 04:00:00 server sshd[2232]: Failed password for user1 from 203.0.113.42 port 22 ssh2
May 15 23:15:00 server sshd[2233]: Failed password for user1 from 10.0.0.5 port 22 ssh2
May 15 23:17:00 server sshd[2234]: Accepted password for user1 from 10.0.0.5 port 22 ssh2
May 15 14:22:01 server sshd[2235]: Failed password for user2 from 192.168.0.20 port 22 ssh2
May 15 14:23:01 server sshd[2236]: Failed password for user2 from 192.168.0.20 port 22 ssh2
"""

pattern = r"(?P<date>\w+ \d+ \d+:\d+:\d+).*sshd.*(?P<result>Failed|Accepted) password for (invalid user )?(?P<user>\S+) from (?P<ip>\d+\.\d+\.\d+\.\d+)"
matches = re.findall(pattern, log_data)

df = pd.DataFrame(matches, columns=['date', 'result', '_', 'user', 'ip']).drop(columns=['_'])
df['timestamp'] = pd.to_datetime("2025 " + df['date'], format='%Y %b %d %H:%M:%S')
df['hour'] = df['timestamp'].dt.hour

display(df)

failed = df[df['result'] == 'Failed']
failed_count = failed.groupby('ip').size().sort_values(ascending=False)
print(failed_count)

off_hours = df[(df['hour'] < 6) | (df['hour'] > 22)]
display(off_hours)

suspicious_ips = failed_count[failed_count >= 2]
print(suspicious_ips)

plt.figure(figsize=(10, 4))
df['hour'].plot.hist(bins=24, rwidth=0.9)
plt.title("Login Attempts by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Number of Attempts")
plt.grid(True)
plt.show()
